![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# M3L2 E16 - Arquitectura de modelos open source: motor, servidor y wrapper (Resolution)

## Qué es este notebook

Hasta ahora usamos siempre un **proveedor comercial** (OpenAI) detrás de `ChatOpenAI`. Este notebook es puramente conceptual: separa las piezas que hacen falta para correr un **modelo open source / open weight** (Llama, Qwen, Mistral, Gemma, DeepSeek...) y compara las alternativas para ejecutarlo. Es la base teórica de E17, donde nos conectamos de verdad a un servidor local.

No requiere API key ni servidor corriendo — es 100% teoría, tablas y diagramas.


In [ ]:
# Este notebook solo necesita langchain-core (ya viene como dependencia de langchain-openai).
# %pip (no !pip) instala siempre en el Python del kernel, aunque tu shell no tenga 'pip' en el PATH.
%pip install langchain-core


## BLOQUE 1 — El problema

Con un proveedor comercial, la arquitectura es simple:

```text
Aplicacion
    ↓
SDK o wrapper
    ↓
API del proveedor
    ↓
Modelo alojado por el proveedor
```

Con un modelo open source, la aplicación tiene que decidir varias cosas que antes resolvía el proveedor:

- Qué modelo usar.
- Dónde ejecutarlo (tu máquina, un servidor propio, un proveedor de inferencia).
- Qué motor de inferencia utilizar.
- Cómo exponerlo a la aplicación (¿un servidor HTTP? ¿carga directa en el proceso?).
- Qué wrapper de LangChain conecta con eso.

```text
Aplicacion
    ↓
LangChain o LangGraph
    ↓
Wrapper o integracion
    ↓
API local o remota
    ↓
Motor de inferencia
    ↓
Modelo
    ↓
CPU, GPU o Apple Silicon
```


## BLOQUE 2 — Definiciones que no hay que confundir

### Modelo

Es el conjunto de parámetros (pesos) aprendidos durante el entrenamiento. Familias conocidas: Llama, Qwen, Mistral, Gemma, Phi, DeepSeek, GPT-OSS.

Un modelo por sí solo **no** es una API, no es LangChain, no administra herramientas ni memoria.

### Modelo base vs modelo instruct

| Tipo | Comportamiento |
|---|---|
| **Base** | Entrenado para continuar texto. `"La IA es..."` → `"...un area de estudio que..."`. No sigue instrucciones de forma confiable. |
| **Instruct / chat** | Ajustado para seguir órdenes y conversar. Es el que querés para casi cualquier aplicación con LangChain. |

Para producción, elegir siempre versiones `Instruct`/`Chat`, y si necesitás tools, versiones ajustadas específicamente para *tool calling*.

### Open source vs open weight

| Término | Qué significa |
|---|---|
| **Open source** | El software se publica con licencia que permite inspección, modificación y redistribución |
| **Open weight** | Los *pesos* del modelo están disponibles, pero la licencia puede tener restricciones y el entrenamiento puede no ser reproducible |

En la práctica, "modelo open source" suele usarse de forma laxa para referirse a cualquier modelo cuyos pesos se pueden descargar y correr localmente.

### Motor de inferencia

Es el software que **ejecuta** el modelo: carga los pesos en RAM/VRAM, tokeniza, corre las operaciones matemáticas, genera tokens. Ejemplos: Ollama, llama.cpp, Hugging Face Transformers, vLLM, Text Generation Inference.

### Servidor de inferencia

Mantiene uno o varios modelos disponibles y expone una API (`POST /v1/chat/completions`). El servidor recibe la petición, corre el motor, devuelve el resultado.

### API compatible con OpenAI

Un servidor "compatible con OpenAI" imita la estructura de endpoints de OpenAI, sin que el modelo sea de OpenAI. Esto permite reutilizar el mismo cliente:

```python
from langchain_openai import ChatOpenAI
```

cambiando solo `base_url`, `api_key` y `model`.

### Wrapper

Adaptador entre LangChain y una API o motor: `ChatOpenAI`, `ChatOllama`, `ChatHuggingFace`, `HuggingFacePipeline`. Transforma mensajes estándar de LangChain al formato del proveedor, y la respuesta de vuelta a `AIMessage`.


### Las capas de una aplicación local

```text
Frontend o interfaz CLI
        ↓
Backend Python
        ↓
LangChain
        ↓
ChatOpenAI (wrapper)
        ↓
LM Studio Server
        ↓
Runtime de inferencia
        ↓
Modelo GGUF
        ↓
CPU, GPU o Apple Silicon
```

| Capa | Responsabilidad |
|---|---|
| Aplicación | Interfaz y lógica de negocio |
| LangChain | Prompts, modelos, parsers, retrievers y agentes |
| Wrapper | Adaptación de mensajes y respuestas |
| Servidor local (LM Studio, etc.) | Gestión y serving local del modelo |
| Motor | Inferencia |
| Modelo | Generación de tokens |
| Hardware | Cálculo y memoria |


## BLOQUE 3 — Siete formas de ejecutar un modelo open source

| Alternativa | Ubicación | Facilidad | Control | Concurrencia | Uso principal |
|---|---|---:|---:|---:|---|
| **LM Studio** | Local | Alta | Medio | Baja | Desarrollo y clase |
| **Ollama** | Local o servidor | Alta | Medio | Baja/Media | Automatización local |
| **Hugging Face Transformers** | Proceso Python | Media | Muy alto | Baja | Investigación / fine-tuning |
| **llama.cpp** | Local o servidor | Media | Alto | Media | GGUF y hardware limitado |
| **vLLM** | Servidor GPU | Media/Baja | Alto | Alta | Producción |
| **Proveedor externo de inferencia** | Nube | Alta | Bajo | Alta | Evitar infraestructura propia |
| **Wrapper propio (`BaseChatModel`)** | Variable | Baja | Muy alto | Variable | API no estándar |

### Regla práctica

```text
Clase o demo               -> LM Studio
Automatizacion / CLI        -> Ollama
Investigacion / fine-tuning -> Transformers
Hardware limitado, GGUF      -> llama.cpp
Produccion, GPU de servidor  -> vLLM
Sin querer administrar GPU   -> proveedor de inferencia
API propia no estandar        -> wrapper personalizado
```


### LM Studio (foco de E17)

- Interfaz gráfica + servidor local + endpoints compatibles con OpenAI.
- Ideal para clases, prototipos y evaluación visual de modelos.
- Limitación: la concurrencia es baja (pensado para uno o pocos usuarios).

```text
LangChain -> ChatOpenAI -> http://localhost:1234/v1 -> LM Studio -> Modelo local
```

### Ollama

- CLI y servicio como experiencia central (en vez de GUI).
- Integración directa: `ChatOllama` (puerto habitual `11434`).

```text
LangChain -> ChatOllama -> Ollama API -> Modelo local
```

| LM Studio | Ollama |
|---|---|
| GUI como experiencia central | CLI y servicio como experiencia central |
| `ChatOpenAI` con `base_url` | `ChatOllama` |
| Puerto habitual `1234` | Puerto habitual `11434` |
| Muy útil para clase visual | Muy útil para automatización |

### vLLM (producción)

```text
LangChain -> ChatOpenAI -> Servidor vLLM -> Modelo en GPU
```

```text
LM Studio -> desarrollo local y exploracion.
vLLM      -> serving productivo y concurrencia.
```

Ambos se consumen desde LangChain con la **misma clase** (`ChatOpenAI` con distinto `base_url`) porque ambos exponen una API compatible con OpenAI.


## BLOQUE 4 — Cuándo crear un wrapper propio

No hace falta escribir un `BaseChatModel` desde cero si el servidor ya soporta una API estándar (OpenAI-compatible, Ollama, Hugging Face Endpoint). Conviene crearlo solo cuando:

- La API no sigue ningún estándar existente.
- Hay autenticación o formato de mensajes propios.
- Se necesitan métricas o callbacks internos específicos.

Esqueleto conceptual (no se ejecuta en esta celda — no hay servidor real detrás):


In [ ]:
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult


class CustomChatModel(BaseChatModel):
    endpoint: str
    api_key: str | None = None

    @property
    def _llm_type(self) -> str:
        return "custom-chat-model"

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any | None = None,
        **kwargs: Any,
    ) -> ChatResult:
        # Aca iria la llamada real a self.endpoint (por ejemplo con httpx.post(...))
        # y la conversion de la respuesta a AIMessage.
        raise NotImplementedError("Esqueleto ilustrativo: no apunta a un servidor real")


print(f"CustomChatModel es subclase de BaseChatModel: {issubclass(CustomChatModel, BaseChatModel)}")


## Resumen — Lo que demuestra E16

```text
Modelo               = pesos entrenados (Llama, Qwen, Mistral, ...)
Motor de inferencia   = software que ejecuta el modelo (Ollama, llama.cpp, vLLM, ...)
Servidor de inferencia = expone el motor via API (POST /v1/chat/completions)
Wrapper               = adaptador entre LangChain y esa API (ChatOpenAI, ChatOllama, ...)
```

**Ninguna de estas piezas es intercambiable por otra** — cada una resuelve un problema distinto, aunque en la práctica varias veces una sola herramienta (como LM Studio) las combine todas menos la última.

**Relacionado con**: E00 - LLM Wrapper, E17 - Conexión a LM Studio.


## Checks automáticos

In [ ]:
def run_checks():
    from langchain_core.language_models.chat_models import BaseChatModel

    assert issubclass(CustomChatModel, BaseChatModel)

    comparativa = {
        "LM Studio": {"ubicacion": "Local", "concurrencia": "Baja"},
        "Ollama": {"ubicacion": "Local o servidor", "concurrencia": "Baja/Media"},
        "Transformers": {"ubicacion": "Proceso Python", "concurrencia": "Baja"},
        "llama.cpp": {"ubicacion": "Local o servidor", "concurrencia": "Media"},
        "vLLM": {"ubicacion": "Servidor GPU", "concurrencia": "Alta"},
    }
    assert len(comparativa) == 5
    assert all("ubicacion" in v and "concurrencia" in v for v in comparativa.values())

    print("M3L2 E16 Resolution checks passed")


run_checks()
